# KeplerGL Interactive Exploration

This notebook provides an interactive exploration of the Global Terrorism Database using KeplerGL.

## Features
- **Point Layer**: Incidents colored by attack type, sized by casualties
- **Hexbin Layer**: 3D extruded hexagons showing incident density
- **Heatmap Layer**: Toggle-able heatmap visualization
- **Time Filter**: Animate through years to see terrorism spread
- **Interactive Filters**: Region, attack type, country

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
from keplergl import KeplerGl

print("KeplerGL loaded successfully!")

## Load Preprocessed Data

In [ ]:
# Define paths
project_dir = Path(os.getcwd()).parent
data_file = project_dir / "data" / "processed" / "gtd_processed.parquet"
config_file = project_dir / "config" / "kepler_config.json"

# Load data
print(f"Loading data from: {data_file}")
df = pd.read_parquet(data_file)
print(f"Loaded {len(df):,} rows")

# Convert datetime for KeplerGL compatibility
df['event_date'] = pd.to_datetime(df['event_date'])

df.head()

In [ ]:
# Load pre-configured settings
with open(config_file, 'r') as f:
    kepler_config = json.load(f)
    
print("Loaded KeplerGL configuration")

## Initialize KeplerGL Map

The map below is fully interactive. You can:
- **Pan/Zoom**: Click and drag, scroll to zoom
- **Rotate**: Right-click and drag
- **Toggle Layers**: Use the layer panel on the left
- **Filter Data**: Use the filter panel to filter by time, region, etc.
- **Animate**: Press play on the time filter to animate over years

In [ ]:
# Create KeplerGL map
map_1 = KeplerGl(
    height=700,
    data={'gtd': df},
    config=kepler_config
)

map_1

## Using the Interactive Map

### Layer Controls (Left Panel)
1. **Points Layer**: Shows individual incidents
   - Color represents attack type
   - Size represents total casualties
   
2. **Hexbin Layer**: 3D aggregation
   - Height shows incident count per hexagon
   - Color intensity shows density
   
3. **Heatmap Layer**: Initially hidden
   - Toggle visibility in layer panel
   - Shows weighted density by casualties

### Time Animation
- Use the timeline slider at the bottom
- Click the play button to animate through time
- Adjust animation speed with the settings

### Filters
- Add filters for `region_txt`, `attacktype1_txt`, `country_txt`
- Combine multiple filters to drill down

In [ ]:
# Save current configuration (after making changes in the UI)
# Uncomment and run after modifying the map

# updated_config = map_1.config
# with open(config_file, 'w') as f:
#     json.dump(updated_config, f, indent=2)
# print("Configuration saved!")

## Export to Standalone HTML

In [ ]:
# Export to HTML
export_dir = project_dir / "exports"
export_dir.mkdir(exist_ok=True)
export_file = export_dir / "gtd_kepler_interactive.html"

map_1.save_to_html(file_name=str(export_file))
print(f"\nExported to: {export_file}")
print(f"File size: {export_file.stat().st_size / (1024*1024):.1f} MB")

## Regional Focus Maps

Create focused views for specific regions.

In [ ]:
# Middle East & North Africa focus
mena_df = df[df['region_txt'] == 'Middle East & North Africa']
print(f"MENA incidents: {len(mena_df):,}")

mena_config = {
    "version": "v1",
    "config": {
        "mapState": {
            "latitude": 30,
            "longitude": 35,
            "zoom": 3.5,
            "pitch": 45,
            "bearing": 0
        },
        "mapStyle": {
            "styleType": "dark"
        }
    }
}

map_mena = KeplerGl(
    height=500,
    data={'mena': mena_df},
    config=mena_config
)

map_mena

In [ ]:
# South Asia focus
sa_df = df[df['region_txt'] == 'South Asia']
print(f"South Asia incidents: {len(sa_df):,}")

sa_config = {
    "version": "v1",
    "config": {
        "mapState": {
            "latitude": 25,
            "longitude": 75,
            "zoom": 3.5,
            "pitch": 45,
            "bearing": 0
        },
        "mapStyle": {
            "styleType": "dark"
        }
    }
}

map_sa = KeplerGl(
    height=500,
    data={'south_asia': sa_df},
    config=sa_config
)

map_sa

## Summary Statistics

In [ ]:
print("="*60)
print("GTD SUMMARY STATISTICS")
print("="*60)
print(f"\nTotal Incidents: {len(df):,}")
print(f"Date Range: {df['iyear'].min()} - {df['iyear'].max()}")
print(f"Countries: {df['country_txt'].nunique()}")
print(f"Total Killed: {df['nkill'].sum():,.0f}")
print(f"Total Wounded: {df['nwound'].sum():,.0f}")
print(f"Perpetrator Groups: {df['gname'].nunique()}")
print("\nTop 5 Countries by Incidents:")
for country, count in df['country_txt'].value_counts().head(5).items():
    print(f"  {country}: {count:,}")